# PROBLEMA: Sistema de Búsqueda de Estudiantes

Tienes un archivo con 10,000 estudiantes (nombre, edad,
promedio). Necesitas implementar un sistema que
permita:
1. Buscar un estudiante por su ID (número de matrícula)
2. Insertar nuevos estudiantes
3. Listar todos los estudiantes en orden por ID

In [1]:
# Importación de herramientas necesarias
import random
import time
import math
from bisect import bisect_right, bisect_left
from faker import Faker

# Entidad Estudiante
class Estudiante:
    def __init__(self, id, nombre, promedio) -> None:
        self.Id = id
        self.Nombre = nombre
        self.Promedio = promedio
    
    def __repr__(self):
        return f"{self.Id} - {self.Nombre} - Promedio: {self.Promedio}"

In [2]:
# Inicializar Faker
fake = Faker('es_MX')

# Creación del Dataset
cantidad_estudiantes = 10000

#Generar IDs únicos y desordenarlos
ids_estudiantes = list(range(1000, 1000 + cantidad_estudiantes))
random.shuffle(ids_estudiantes)

estudiantes = []

for est_id in ids_estudiantes:
    estudiante = Estudiante(id=est_id, nombre=fake.name(), promedio= round(random.uniform(0, 5), 1))
    estudiantes.append(estudiante)

In [3]:
# Implementación con lista nativa

class ListaEstudiantes:
    def __init__(self):
        self.estudiantes = []
    
    def insertar(self, estudiante):
        self.estudiantes.append(estudiante)
    
    def buscar(self, id):
        for est in self.estudiantes:
            if est.Id == id:
                return est
        return None
    
    def listar_ordenado(self):
        return sorted(self.estudiantes, key= lambda est: est.Id)

In [4]:
# Implementación con ABB co-generado con copilot

class NodoABB:
    def __init__(self, estudiante) -> None:
        self.estudiante = estudiante
        self.izq = None
        self.der = None

class ABB:
    def __init__(self) -> None:
        self.root = None
    
    def insertar(self, estudiante):
        nuevo_nodo = NodoABB(estudiante)

        # Si el árbol está vacío
        if self.root is None:
            self.root = nuevo_nodo
            return
        
        nodo_actual = self.root

        while True:
            # No se insertan IDs duplicados, en este caso se actualiza el estudiante
            if estudiante.Id == nodo_actual.estudiante.Id:
                nodo_actual.estudiante = estudiante
                return
            # Ir hacia la izquierda
            if estudiante.Id < nodo_actual.estudiante.Id:
                if nodo_actual.izq is None:
                    nodo_actual.izq = nuevo_nodo
                    return

                nodo_actual = nodo_actual.izq
            # Ir hacia la derecha
            else:
                if nodo_actual.der is None:
                    nodo_actual.der = nuevo_nodo
                    return

                nodo_actual = nodo_actual.der
    
    def buscar(self, id):
        actual = self.root
        while actual is not None:
            if id == actual.estudiante.Id:
                return actual.estudiante
            if id < actual.estudiante.Id:
                actual = actual.izq
            elif id > actual.estudiante.Id:
                actual = actual.der
        return None
    
    def listar_ordenado(self):
        resultado = []
        pila = []
        nodo_actual = self.root

        # Recorrido inorden iterativo
        while pila or nodo_actual is not None:
            while nodo_actual is not None:
                pila.append(nodo_actual)
                nodo_actual = nodo_actual.izq

            nodo_actual = pila.pop()
            resultado.append(nodo_actual.estudiante)

            nodo_actual = nodo_actual.der

        return resultado
    

La siguiente implementación del árbol B+ fue ajustada para el ejercicio a partir de la implementación publicada en [programiz](https://www.programiz.com/dsa/b-plus-tree).

### Ajustes realizados

- Se cambió la comparación de los valores de tipo `string` a valores de tipo `int`, debido a que los IDs utilizados son números enteros y deben compararse numéricamente
- Se eliminó la agrupación de múltiples claves por valor, ya que cada estudiante posee un ID único
- Se agregó el método `buscar(id_estudiante)`, que devuelve directamente el estudiante asociado al ID indicado
- Se agregó el método `listar_ordenado()`, que recorre las hojas enlazadas mediante `nextKey` y devuelve los estudiantes ordenados por ID


In [5]:
# Implementación con Árbol B+

class NodoBplus:
    def __init__(self, order):
        self.order = order
        self.values = [] # En hojas: IDs; en nodos internos: separadores
        self.keys = [] # En hojas: estudiantes; en nodos internos: hijos
        self.nextKey = None # Enlace entre hojas
        self.parent = None
        self.checkLeaf = False

    def insert_at_leaf(self, leaf, value, estudiante):
        if self.values:
            for i in range(len(self.values)):
                if value == self.values[i]:
                    # Si el ID ya existes se actualiza el estudiante
                    self.keys[i] = estudiante
                    return
                elif value < self.values[i]:
                    self.values.insert(i, value)
                    self.keys.insert(i, estudiante)
                    return
                elif i+ 1 == len(self.values):
                    self.values.append(value)
                    self.keys.append(estudiante)
                    return
        else:
            self.values = [value]
            self.keys = [estudiante]

class BplusTree:
    def __init__(self, order) -> None:
        self.root = NodoBplus(order)
        self.root.checkLeaf = True
    
    def insertar(self, estudiante):
        value = estudiante.Id
        old_node = self.search(value)
        old_node.insert_at_leaf(old_node, value, estudiante)
        
        if len(old_node.values) == old_node.order:
            node1 = NodoBplus(old_node.order)
            node1.checkLeaf = True
            node1.parent = old_node.parent
            
            mid = int(math.ceil(old_node.order/2))-1
            
            # Parte derecha se mueve a la nueva hoja
            node1.values = old_node.values[mid + 1:]
            node1.keys = old_node.keys[mid + 1:]
            node1.nextKey = old_node.nextKey

            # Parte izquierda se queda en la hoja vieja
            old_node.values = old_node.values[:mid + 1]
            old_node.keys = old_node.keys[:mid + 1]
            old_node.nextKey = node1
            
            self.insert_in_parent(old_node, node1.values[0], node1)
    
    def search(self, value):
        current_node = self.root

        while current_node.checkLeaf == False:
            # bisect_right nos da directamente el índice del puntero hijo que debemos seguir
            i = bisect_right(current_node.values, value)
            current_node = current_node.keys[i]

        return current_node
    
    def buscar(self, id):
        node = self.search(id) # Hoja donde deberia encontrarse
        # Buscamos la posición exacta del ID dentro de la hoja
        i = bisect_left(node.values, id)
        
        if i < len(node.values) and node.values[i] == id:
            return node.keys[i]
        
        return None
    
    def listar_ordenado(self):
        resultado = []

        nodo = self.root
        while nodo.checkLeaf == False:
            nodo = nodo.keys[0]

        while nodo is not None:
            for estudiante in nodo.keys:
                resultado.append(estudiante)
            nodo = nodo.nextKey

        return resultado
    
    def insert_in_parent(self, n, value, ndash):
        if self.root == n:
            rootNode = NodoBplus(n.order)
            rootNode.values = [value]
            rootNode.keys = [n, ndash]
            self.root = rootNode
            n.parent = rootNode
            ndash.parent = rootNode
            return

        parentNode = n.parent

        for i in range(len(parentNode.keys)):
            if parentNode.keys[i] == n:
                parentNode.values = parentNode.values[:i] + [value] + parentNode.values[i:]
                parentNode.keys = parentNode.keys[:i + 1] + [ndash] + parentNode.keys[i + 1:]

                if len(parentNode.keys) > parentNode.order:
                    parentdash = NodoBplus(parentNode.order)
                    parentdash.parent = parentNode.parent

                    mid = int(math.ceil(parentNode.order / 2)) - 1

                    parentdash.values = parentNode.values[mid + 1:]
                    parentdash.keys = parentNode.keys[mid + 1:]

                    value_ = parentNode.values[mid]

                    if mid == 0:
                        parentNode.values = parentNode.values[:mid + 1]
                    else:
                        parentNode.values = parentNode.values[:mid]

                    parentNode.keys = parentNode.keys[:mid + 1]

                    for j in parentNode.keys:
                        j.parent = parentNode

                    for j in parentdash.keys:
                        j.parent = parentdash

                    self.insert_in_parent(parentNode, value_, parentdash)

                return

In [6]:
# Construción de estructuras llenadas con el dataset generado en principio
def construir_estructuras(estudiantes):
    lista = ListaEstudiantes()
    abb = ABB()
    abplus = BplusTree(order=8)
    for estudiante in estudiantes:
        lista.insertar(estudiante)
        abb.insertar(estudiante)
        abplus.insertar(estudiante)
    
    return lista, abb, abplus

lista, abb, abplus = construir_estructuras(estudiantes)

Bloque de pruebas y comparación de tiempos de busqueda generado con Gemini 3.1 Pro

In [7]:
def comparar_tiempos_busqueda(lista, abb, abplus, estudiantes, cantidad_buscar = 500, repeticiones=50):
    # Extraemos IDs reales para garantizar que la búsqueda recorra las estructuras
    todos_los_ids = [est.Id for est in estudiantes]
    ids_muestra = random.sample(todos_los_ids, cantidad_buscar)
    
    print(f"--- Comparativa de Rendimiento ---")
    print(f"Buscando {cantidad_buscar} estudiantes | Promedio de {repeticiones} repeticiones\n")

    estructuras = [
        ("Lista Nativa", lista.buscar),
        ("Árbol ABB", abb.buscar),
        ("Árbol B+", abplus.buscar)
    ]
    
    for nombre, funcion_busqueda in estructuras:
        tiempos = []
        
        for _ in range(repeticiones):
            inicio = time.perf_counter()
            
            # Ejecutamos el lote de búsquedas
            for id_estudiante in ids_muestra:
                funcion_busqueda(id_estudiante)
                
            fin = time.perf_counter()
            tiempos.append(fin - inicio)
            
        # Cálculo del promedio
        tiempo_promedio = sum(tiempos) / repeticiones
        
        # Formateo alineado para facilitar la lectura
        print(f"{nombre:<15} : {tiempo_promedio:.6f} segundos por lote")

# Ejecutar la prueba pasando las estructuras ya construidas
comparar_tiempos_busqueda(lista, abb, abplus, estudiantes)

--- Comparativa de Rendimiento ---
Buscando 500 estudiantes | Promedio de 50 repeticiones

Lista Nativa    : 0.098899 segundos por lote
Árbol ABB       : 0.000649 segundos por lote
Árbol B+        : 0.000578 segundos por lote


In [8]:
prueba_id = estudiantes[42].Id

print(f"PRUEBA DE BÚSQUEDA (ID: {prueba_id})")
print("En Lista Nativa :", lista.buscar(prueba_id))
print("En Árbol ABB    :", abb.buscar(prueba_id))
print("En Árbol B+     :", abplus.buscar(prueba_id))

print(f"PRUEBA DE RECORRIDO EN B+")
primeros_5 = abplus.listar_ordenado()[:5]
for est in primeros_5:
    print(est)
    
print(f"PRUEBA DE INSERCIÓN EN B+")
nuevo_id = 99999
estudiante_nuevo = Estudiante(id=nuevo_id, nombre="Ada Lovelace", promedio=5.0)

abplus.insertar(estudiante_nuevo)
print("Buscando estudiante recién insertado:", abplus.buscar(nuevo_id))

PRUEBA DE BÚSQUEDA (ID: 6272)
En Lista Nativa : 6272 - Lic. Elena Camarillo - Promedio: 2.3
En Árbol ABB    : 6272 - Lic. Elena Camarillo - Promedio: 2.3
En Árbol B+     : 6272 - Lic. Elena Camarillo - Promedio: 2.3
PRUEBA DE RECORRIDO EN B+
1000 - Fabiola Raya Rael - Promedio: 3.8
1001 - Hilda Vanesa Carreón Berríos - Promedio: 0.4
1002 - Ricardo Felipe de la Torre Mota - Promedio: 0.4
1003 - Lic. Víctor Cano - Promedio: 3.7
1004 - Elvira Virginia Bustamante - Promedio: 0.4
PRUEBA DE INSERCIÓN EN B+
Buscando estudiante recién insertado: 99999 - Ada Lovelace - Promedio: 5.0
